# Order Estimation with simulated data

Simulate data, fit different HMMs with a varying number of hidden states. Compute BIC and AIC, explore dependence on sample size and difference between AR(1) model parameters.

## Imports

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from methods.hmm_ar_1_k_states import (
    simulate_rs_ar1,
    transform_params,
    obs_density,
    forward_algorithm,
    neg_loglik,
    fit_model,
    filtered_probs
)

## Simulate data

In [2]:
T = 1000
seed = 1
# AR(1) parameters
beta = np.array([0.2, 0.8])
sigma = np.array([0.5, 1.0])

# transition matrix
P = np.array([
    [0.95, 0.05],
    [0.05, 0.95]
])


y, states = simulate_rs_ar1(T=T, beta=beta, sigma=sigma, P=P, seed=seed)

print("")

## HMM modell med ulikt antall states

Estimerer HMM med 1 til 4 states. Lagrer AIC og BIC, samt likelihood. Bruker guess nær sanne parameter der det er mulig for raskere kode. Printer fremgang

In [3]:
K_vals = [1, 2, 3, 4]
results_all = []

for k in K_vals:
    print(f"\nStarter K = {k}")
    
    best_loglik = -np.inf
    best_result = None
    best_params = None

    if k == 2:
        beta0 = np.log((1 + beta) / (1 - beta)) + np.random.normal(scale=0.1, size=2)
        sigma0 = np.log(sigma) + np.random.normal(scale=0.1, size=2)
        P0 = np.log(P + 1e-8) + np.random.normal(scale=0.1, size=(2, 2))
    else:
        beta0 = np.random.uniform(-0.9, 0.9, size=k)
        sigma0 = np.random.normal(size=k)
        P0 = np.random.normal(size=(k, k))

    result, params_hat = fit_model(y, beta0, sigma0, P0)
    loglik = -result.fun

    print(f"K={k}, loglik={loglik:.2f}")

    if loglik > best_loglik:
        best_loglik = loglik
        best_result = result
        best_params = params_hat

    n_params = 2*k + k*(k-1)
    n = len(y)

    aic = -2*best_loglik + 2*n_params
    bic = -2*best_loglik + np.log(n)*n_params

    print(f"Best for K={k}: AIC={aic:.2f}, BIC={bic:.2f}")

    results_all.append({
        "K": k,
        "loglik": best_loglik,
        "AIC": aic,
        "BIC": bic,
        "result": best_result,
        "params": best_params
    })


Starter K = 1
K=1, loglik=-1181.47
Best for K=1: AIC=2366.94, BIC=2376.76

Starter K = 2
K=2, loglik=-1103.58
Best for K=2: AIC=2219.15, BIC=2248.60

Starter K = 3
K=3, loglik=-1099.03
Best for K=3: AIC=2222.05, BIC=2280.94

Starter K = 4
K=4, loglik=-1097.68
Best for K=4: AIC=2235.37, BIC=2333.52


# Undersøke hvordan sample size påvirker

In [4]:
sample_sizes = [200, 500, 1000, 2000]
n_rep = 2
K_vals = [1, 2, 3]

order_results = []

for T in sample_sizes:
    print(f"\n===== Sample size T = {T} =====")
    
    aic_choices = []
    bic_choices = []

    for rep in range(n_rep):
        print(f"Repetisjon {rep+1}/{n_rep}")
        
        # Simuler data fra sann 2-state modell
        y, states = simulate_rs_ar1(
            T=T,
            beta=beta,
            sigma=sigma,
            P=P,
            seed=1000 + rep
        )

        results_all = []

        for k in K_vals:
            print(f"  Estimerer K = {k}")
            
            if k == 2:
                beta0 = np.log((1 + beta) / (1 - beta)) + np.random.normal(scale=0.1, size=2)
                sigma0 = np.log(sigma) + np.random.normal(scale=0.1, size=2)
                P0 = np.log(P + 1e-8) + np.random.normal(scale=0.1, size=(2, 2))
            else:
                beta0 = np.random.uniform(-0.9, 0.9, size=k)
                sigma0 = np.random.normal(size=k)
                P0 = np.random.normal(size=(k, k))

            result, params_hat = fit_model(y, beta0, sigma0, P0)
            loglik = -result.fun

            n_params = 2*k + k*(k-1)
            aic = -2*loglik + 2*n_params
            bic = -2*loglik + np.log(len(y))*n_params

            print(f"    loglik={loglik:.2f}, AIC={aic:.2f}, BIC={bic:.2f}")

            results_all.append({
                "K": k,
                "loglik": loglik,
                "AIC": aic,
                "BIC": bic,
                "result": result,
                "params": params_hat
            })

        best_k_aic = min(results_all, key=lambda d: d["AIC"])["K"]
        best_k_bic = min(results_all, key=lambda d: d["BIC"])["K"]

        aic_choices.append(best_k_aic)
        bic_choices.append(best_k_bic)

        print(f"  Valgt av AIC: K={best_k_aic}")
        print(f"  Valgt av BIC: K={best_k_bic}")

    order_results.append({
        "T": T,
        "AIC_correct_rate": np.mean(np.array(aic_choices) == 2),
        "BIC_correct_rate": np.mean(np.array(bic_choices) == 2),
        "AIC_choices": aic_choices,
        "BIC_choices": bic_choices
    })

print("\n===== Oppsummering =====")
for res in order_results:
    print(
        f"T={res['T']}: "
        f"AIC velger K=2 i {res['AIC_correct_rate']:.2f} av simuleringene, "
        f"BIC velger K=2 i {res['BIC_correct_rate']:.2f} av simuleringene"
    )


===== Sample size T = 200 =====
Repetisjon 1/2
  Estimerer K = 1
    loglik=-230.15, AIC=464.29, BIC=470.89
  Estimerer K = 2
    loglik=-207.40, AIC=426.81, BIC=446.60
  Estimerer K = 3
    loglik=-204.16, AIC=432.32, BIC=471.90
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 2/2
  Estimerer K = 1
    loglik=-273.06, AIC=550.13, BIC=556.72
  Estimerer K = 2
    loglik=-259.67, AIC=531.34, BIC=551.13
  Estimerer K = 3
    loglik=-258.48, AIC=540.97, BIC=580.55
  Valgt av AIC: K=2
  Valgt av BIC: K=2

===== Sample size T = 500 =====
Repetisjon 1/2
  Estimerer K = 1
    loglik=-614.37, AIC=1232.74, BIC=1241.17
  Estimerer K = 2
    loglik=-582.00, AIC=1175.99, BIC=1201.28
  Estimerer K = 3
    loglik=-580.75, AIC=1185.50, BIC=1236.08
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 2/2
  Estimerer K = 1
    loglik=-640.04, AIC=1284.08, BIC=1292.51
  Estimerer K = 2
    loglik=-599.50, AIC=1211.00, BIC=1236.29
  Estimerer K = 3
    loglik=-596.44, AIC=1216.89, BIC=1267.46
  Valgt av

# Litt prompting

In [5]:
beta1 = 0.2
beta2_vals = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

sigma = np.array([0.5, 1.0])

P = np.array([
    [0.95, 0.05],
    [0.05, 0.95]
])

T = 1000
K_vals = [1, 2, 3]

diff_results = []

for beta2 in beta2_vals:
    beta = np.array([beta1, beta2])
    diff = abs(beta2 - beta1)

    print(f"\n===== beta difference = {diff:.2f} =====")
    print(f"beta = {beta}")

    y, states = simulate_rs_ar1(
        T=T,
        beta=beta,
        sigma=sigma,
        P=P,
        seed=1
    )

    results_all = []

    for k in K_vals:
        print(f"  Estimerer K={k}")

        if k == 2:
            beta0 = np.log((1 + beta) / (1 - beta))
            sigma0 = np.log(sigma)
            P0 = np.log(P + 1e-8)
        else:
            beta0 = np.random.uniform(-0.9, 0.9, size=k)
            sigma0 = np.random.normal(size=k)
            P0 = np.random.normal(size=(k, k))

        result, params_hat = fit_model(y, beta0, sigma0, P0)
        loglik = -result.fun

        n_params = 2*k + k*(k-1)
        aic = -2*loglik + 2*n_params
        bic = -2*loglik + np.log(len(y))*n_params

        print(f"    loglik={loglik:.2f}, AIC={aic:.2f}, BIC={bic:.2f}")

        results_all.append({
            "K": k,
            "AIC": aic,
            "BIC": bic
        })

    best_k_aic = min(results_all, key=lambda d: d["AIC"])["K"]
    best_k_bic = min(results_all, key=lambda d: d["BIC"])["K"]

    print(f"  → AIC velger K={best_k_aic}")
    print(f"  → BIC velger K={best_k_bic}")

    diff_results.append({
        "beta_diff": diff,
        "best_k_aic": best_k_aic,
        "best_k_bic": best_k_bic
    })


===== beta difference = 0.10 =====
beta = [0.2 0.3]
  Estimerer K=1
    loglik=-1141.88, AIC=2287.76, BIC=2297.58
  Estimerer K=2
    loglik=-1098.88, AIC=2209.76, BIC=2239.21
  Estimerer K=3
    loglik=-1095.45, AIC=2214.91, BIC=2273.80
  → AIC velger K=2
  → BIC velger K=2

===== beta difference = 0.20 =====
beta = [0.2 0.4]
  Estimerer K=1
    loglik=-1143.46, AIC=2290.93, BIC=2300.74
  Estimerer K=2
    loglik=-1098.15, AIC=2208.29, BIC=2237.74
  Estimerer K=3
    loglik=-1094.94, AIC=2213.87, BIC=2272.76
  → AIC velger K=2
  → BIC velger K=2

===== beta difference = 0.30 =====
beta = [0.2 0.5]
  Estimerer K=1
    loglik=-1147.38, AIC=2298.76, BIC=2308.58
  Estimerer K=2
    loglik=-1098.22, AIC=2208.44, BIC=2237.88
  Estimerer K=3
    loglik=-1094.34, AIC=2212.69, BIC=2271.58
  → AIC velger K=2
  → BIC velger K=2

===== beta difference = 0.40 =====
beta = [0.2 0.6]
  Estimerer K=1
    loglik=-1154.18, AIC=2312.37, BIC=2322.18
  Estimerer K=2
    loglik=-1099.10, AIC=2210.21, BIC=